# Using RiskScape with Jupyter Notebook

RiskScape models are normally defined by files stored on your local system. We can integrate with the Jupyter notebook environment by saving cells to file (with the `%%writefile` magic command), and once the model has run, using some basic python to load up the output files for display.


## Project files

In [ ]:
%%writefile project.ini
[project]
description = Example models to determine buildings in south-east Upolu exposed to inundation, \
    based on data that models the 2009 tsunami

[model building-damage]
description = Model that calculates building damage
framework = pipeline
location = building-damage-pipeline.txt

[type damage_states]
type.DS_1 = floating
type.DS_2 = floating
type.DS_3 = floating
type.DS_4 = floating
type.DS_5 = floating

[function Samoa_Building_Fragility]
description = Samoa tsunami fragility functions for buildings
location = Samoa_Building_Fragility.py
argument-types = [building: struct(Cons_Frame: text), hazard: nullable(floating)]
return-type = damage_states
framework = jython

In [ ]:
%%writefile building-damage-pipeline.txt
input(relation: 'data/Buildings_SE_Upolu.shp', name: 'exposure') as exposures_input
 -> select({exposure}) as exposures
 -> join(on: true) as exposures_join_hazards
 -> select({*, sample_closest(geometry: exposure, coverage: hazard) as hazard}) as "sample_hazard_layer"
 -> join(on: true) as exposures_join_areas
 -> select({*, sample_closest(geometry: exposure, coverage: area, buffer-distance: 1000) as area})
 -> select({*, probability: map(hazard, hv -> Samoa_Building_Fragility(exposure, hv))})
 # randomly assign a damage state based on the probabilities, i.e. likelihood of being in
 # one damage state and not others
 -> select({*, if_then_else(is_null(hazard), '0: Not damaged',
                            random_choice(['0: Not damaged', '1: Light', '2: Minor', '3: Moderate', '4: Severe', '5: Collapse'],
                              weights: [1.0 - probability.DS_1,
                                        probability.DS_1 - probability.DS_2,
                                        probability.DS_2 - probability.DS_3,
                                        probability.DS_3 - probability.DS_4,
                                        probability.DS_4 - probability.DS_5,
                                        probability.DS_5])) as DamageState})
 -> select({*}) as event_impact_table

input(value: bookmark('data/MaxEnv_All_Scenarios_50m.tif'), name: 'hazard') as hazards_input
 -> select({hazard as hazard}) as hazards
 -> exposures_join_hazards.rhs

input(relation: 'data/Samoa_constituencies.shp', name: 'area') as areas_input
 -> group({to_coverage(area) as area}) as areas
 -> exposures_join_areas.rhs

event_impact_table
 -> select({*}) as "report_event-impact"
 -> group(by: {area.Region as Region}, select: {Region, count(hazard) as Number_Exposed, count(exposure) as Total_buildings, bucket(pick: b -> b = DamageState, select: {count(*)}, buckets: {None: '0: Not damaged', Light: '1: Light', Minor: '2: Minor', Moderate: '3: Moderate', Severe: '4: Severe', Collapse: '5: Collapse'}) as Damage})
 -> sort([Region], direction: ['ASC'])
 -> save(name: 'summary', format: 'csv') as save_summary
 
 event_impact_table
 -> save(name: 'event-impact') as save_event_impact

event_impact_table
 -> group(by: {area as Region}, select: {Region, count(hazard) as Number_Exposed, count(exposure) as Total_buildings, bucket(pick: b -> b = DamageState, select: {count(*)}, buckets: {None: '0: Not damaged', Light: '1: Light', Minor: '2: Minor', Moderate: '3: Moderate', Severe: '4: Severe', Collapse: '5: Collapse'}) as Damage})
  -> save(name: 'regional-impact', format: 'geojson') as save_regional_impact

In [ ]:
%%writefile Samoa_Building_Fragility.py
# Note: this function was provided by Earth Sciences New Zealand https://earthsciences.nz and has been
# refactored and adapted for this tutorial.
def function(building, hazard_depth):
    DS_1_Prob = 0.0
    DS_2_Prob = 0.0
    DS_3_Prob = 0.0
    DS_4_Prob = 0.0
    DS_5_Prob = 0.0
    construction = building["Cons_Frame"]

    if hazard_depth is not None and hazard_depth > 0:
        DS_1_Prob = log_normal_cdf(hazard_depth, -0.53, 0.46)        
    
        if construction in ['Masonry', 'Steel']:
            DS_2_Prob = log_normal_cdf(hazard_depth, -0.33, 0.4)
            DS_3_Prob = log_normal_cdf(hazard_depth, 0.1, 0.35)
            DS_4_Prob = log_normal_cdf(hazard_depth, 0.26, 0.41)
            DS_5_Prob = log_normal_cdf(hazard_depth, 0.39, 0.4)
        elif construction in ['Reinforced_Concrete', 'Reinforced Concrete']:
            DS_2_Prob = log_normal_cdf(hazard_depth, -0.33, 0.4)
            DS_3_Prob = log_normal_cdf(hazard_depth, 0.13, 0.56)
            DS_4_Prob = log_normal_cdf(hazard_depth, 0.53, 0.54)
            DS_5_Prob = log_normal_cdf(hazard_depth, 0.86, 0.94)
        else: # 'Timber' or unknown
            DS_2_Prob = log_normal_cdf(hazard_depth, -0.33, 0.4)
            DS_3_Prob = log_normal_cdf(hazard_depth, 0.06, 0.38)
            DS_4_Prob = log_normal_cdf(hazard_depth, 0.1, 0.4)
            DS_5_Prob = log_normal_cdf(hazard_depth, 0.1, 0.28)

    result = {}
    result['DS_1'] = DS_1_Prob
    result['DS_2'] = DS_2_Prob
    result['DS_3'] = DS_3_Prob
    result['DS_4'] = DS_4_Prob
    result['DS_5'] = DS_5_Prob
    return result

def log_normal_cdf(x, mean, stddev):
    # this uses the built-in RiskScape 'lognorm_cdf' function
    return functions.get('lognorm_cdf').call(x, mean, stddev)

## Running RiskScape

You can run RiskScape commands in Jupyter (or any other shell command for that matter) by prefixing the command with an exclamation mark. 
Jupyter does not handle RiskScape's progress output particularly well, so we recommend disabling it by adding `--progress-indicator=none` to your command. 

In this example, we've also added `--output=output` and `--replace` so that RiskScape saves the outputs in the same folder each time (rather than creating new folders with timestamps. This makes it easier to load these outputs back into RiskScape later. If you want to keep older model runs, you may wish to remove these flags. 

In [ ]:
!riskscape model run --progress-indicator=none --output=output --replace building-damage

## Displaying Results
CSV results can be displayed by using `pandas`.

Geospatial data can be displayed using `geopandas`, although some formatting may be required.

In [ ]:
import pandas as pd
pd.read_csv('output/summary.csv')

In [ ]:
%matplotlib inline
import geopandas as gpd
data = gpd.read_file("output/regional-impact.geojson")
data.plot(column='Damage.Collapse.count', cmap='Reds', legend=True)

## Displaying Results (continued)

You can also use `matplotlib` to graph your results. Below are a couple of examples that:
- create a bar graph for the total buildings in each damage state
- create a curve (scatter plot) of the hazard depth vs DS5 probability relationship for masonry buildings

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
df = pd.read_csv('output/summary.csv')
states = ['Light', 'Minor', 'Moderate', 'Severe', 'Collapse']
plt.bar(states, [ df['Damage.' + s + '.count'].sum() for s in states])

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import geopandas as gpd
df = gpd.read_file('output/event-impact.shp')
df = df[(df.hazard > 0) & (df.Cons_Frame == 'Masonry')]
plt.title('Probability of Collapsed Masonry buildings')
plt.ylabel('Probability')
plt.xlabel('Inundation depth (m)')
plt.scatter(df['hazard'], df['DS_5'])

## Running RiskScape alternative

Below is an alternative way of running a RiskScape model from within Python.
This approach can be handy when batching up model runs, e.g. looping over several different models or parameters and running them all at once.

In [ ]:
import os
MODEL = 'building-damage'
os.system('riskscape model run --output=output --replace ' + MODEL)